In [ ]:
# Imports
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("/projeto")

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import explode
from conf.spark_session import get_spark_session
from delta.tables import DeltaTable
from datetime import datetime

# Sessão do Spark com Delta Lake
spark = get_spark_session()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Função para visualização dos dados
def show_df(df, n=5):
    display(df.limit(n).toPandas())

In [ ]:
# Caminho das ingestões 
base_path = "s3a://datalake/bronze/raw"

# Função para listar os diretórios
def list_dirs(path):
    hadoop_conf = spark._jsc.hadoopConfiguration()
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), hadoop_conf
    )
    status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(path))

    return [
        f.getPath().toString()
        for f in status
        if f.isDirectory()
    ]

# Função para buscar o diretório da última execução
def get_latest_execution_path(base_path):

    ingestion_dirs = list_dirs(base_path)

    if not ingestion_dirs:
        raise Exception("Nenhuma ingestion_date encontrada.")

    latest_ingestion = sorted(ingestion_dirs)[-1]

    execution_dirs = list_dirs(latest_ingestion)

    if not execution_dirs:
        raise Exception("Nenhum execution_id encontrado.")

    latest_execution = sorted(execution_dirs)[-1]

    return latest_ingestion, latest_execution


latest_ingestion, latest_execution = get_latest_execution_path(base_path)

print(latest_ingestion)
print(latest_execution)

In [ ]:
# Montando o path
path = latest_execution

# Lendo os dados com Spark
df = (
    spark.read
    .option("multiline", "true")
    .json(path)
)

In [ ]:
# Adicionando as colunas para governança
df_bronze = (
    df
    .withColumn("ingestion_date", F.lit(ingestion_date))
    .withColumn("execution_id", F.lit(execution_id))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

In [ ]:
# Lendo a tabela de controle de execução do Hive com Spark
df_controle_execucao = spark.table("lakehouse.controle_execucao")
show_df(df_controle_execucao)

In [ ]:
# Buscar a execução pendente na tabela de controle de execução
df_execucao = spark.sql("""
    SELECT execution_id
    FROM lakehouse.controle_execucao
    WHERE processed = false
    ORDER BY created_at
    LIMIT 1
""")

df_execucao.createOrReplaceTempView("df_execucao")
show_df(df_execucao)

In [ ]:
if df_execucao.rdd.isEmpty():
    print("Nenhuma execução pendente para processamento.")
    raise SystemExit()

In [ ]:
# Salvando o id de execução pendende em uma variavel
execution_id = df_execucao.collect()[0]["execution_id"]
print(id_execucao)

# Definindo a data de ingestão
ingestion_date = datetime.strptime(execution_id[:8], "%Y%m%d").strftime("%Y-%m-%d")
print(ingestion_date)

In [ ]:
# Montando o path para ingestão
raw_path = f"""
s3a://datalake/bronze/raw/
ingestion_date={ingestion_date}/
execution_id={execution_id}
""".replace("\n","")

print(raw_path)

In [ ]:
# Ler os arquivos da última execução
df_raw = (
    spark.read
    .option("multiline", True)
    .json(raw_path)
)

In [ ]:
# Adicionando colunas para controle de ingestão
df_bronze = (
    df_raw
    .withColumn("ingestion_date", F.lit(ingestion_date))
    .withColumn("execution_id", F.lit(execution_id))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

show_df(df_bronze)

In [ ]:
# Verificando o schema do dataframe
df_bronze.printSchema()

In [ ]:
# Contagem de registros
df_bronze.count()

In [ ]:
# Definindo o path na camada bronze
bronze_path = "s3a://datalake/bronze/pacientes"

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .option("mergeSchema", "true")
    .save(bronze_path)
)

In [ ]:
bronze_path = "s3a://datalake/bronze/pacientes"
df_pacientes = spark.read.format("delta").load(bronze_path)
show_df(df_pacientes)

In [ ]:
# Atualiuzando a tabela de controle de execução
spark.sql(f"""
    UPDATE lakehouse.controle_execucao
    SET
        processed = true,
        processed_timestamp = current_timestamp()
    WHERE execution_id = '{execution_id}'
""")

In [ ]:
# Lendo a tabela controle de execução do Hive com Spark
df_atualizado = spark.table("lakehouse.controle_execucao")

show_df(df_atualizado)

#df_atualizado.show(vertical=True)